In [11]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, ModelProvider
from agents.tracing import set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(True)

In [12]:
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",  # Ollama's OpenAI-compatible endpoint
    api_key="ollama",  # Dummy key — Ollama doesn't need one
)

# Step 3: Wrap it in the SDK's model class
local_model = OpenAIChatCompletionsModel(
    model="minimax-m2:cloud",  # Must match what you pulled with `ollama pull`
    openai_client=ollama_client,
)

# Step 4: Create agent with local model
agent = Agent(
    name="Local Assistant",
    instructions="You are a helpful assistant running locally via Ollama. Be concise.",
    model=local_model,  # <-- This is the key line!
)

# Step 5: Run it
result = await Runner.run(agent, "What is the Agents SDK in one paragraph?")
print(result.final_output)

The **Agents SDK** (typically referring to OpenAI's Agents SDK) is a Python framework for building AI agents that can use tools, maintain memory, and handle multi-step workflows. It provides primitives like `Agent`, `Runner`, and `Tool` for creating autonomous systems that can reason, execute code, search, and interact with external services. It differs from the older "Assistant API" by being more lightweight and focused on custom agent architectures rather than a fully managed solution.


In [13]:
class OllamaProvider(ModelProvider):
    """Custom ModelProvider that routes all model requests to Ollama."""

    def __init__(self, model_name: str = local_model):
        self.model_name = model_name
        self.client = AsyncOpenAI(base_url=ollama_client, api_key=ollama_client)

    def get_model(self, model_name: str | None = None) -> Model:
        return OpenAIChatCompletionsModel(
            model=model_name or self.model_name, openai_client=self.client
        )


# Now create agents WITHOUT specifying model each time
ollama = OllamaProvider()

agent = Agent(
    name="Local Agent",
    instructions="You are a local AI assistant. Be helpful and concise.",
    model=ollama.get_model(),  # Clean and reusable!
)

result = await Runner.run(agent, "Explain what Ollama is in 2 sentences.")
print(result.final_output)

NameError: name 'Model' is not defined

In [18]:
agent = Agent(
    name="LiteLLM Agent",
    instructions="You are a helpful assistant.",
    # IMPORTANT: prepend 'ollama_chat/' — tells LiteLLM to use Ollama's chat API
    model=LitellmModel(model="minimax-m2:cloud", base_url="http://localhost:11434/v1"),
)

result = await Runner.run(agent, "Hello! What model are you?")
print(result.final_output)


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



BadRequestError: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=minimax-m2:cloud
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers